In [9]:
#imports

import pandas as pd
import numpy as np
import os

# Paths
DATA_DIR = "C:/Users/Aishwarya/Desktop/job-market-project/"
OUTPUT_DIR = "C:/Users/Aishwarya/Desktop/job-market-project/output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [11]:
#Load all files
print("Loading files...")

postings = pd.read_csv(DATA_DIR + "postings.csv", low_memory=False)
companies = pd.read_csv(DATA_DIR + "companies.csv")
job_skills = pd.read_csv(DATA_DIR + "job_skills.csv")
skills = pd.read_csv(DATA_DIR + "skills.csv")
job_industries = pd.read_csv(DATA_DIR + "job_industries.csv")
industries = pd.read_csv(DATA_DIR + "industries.csv")
salaries = pd.read_csv(DATA_DIR + "salaries.csv")
benefits = pd.read_csv(DATA_DIR + "benefits.csv")
company_industries = pd.read_csv(DATA_DIR + "company_industries.csv")

print(f"Postings loaded: {len(postings):,} rows")

Loading files...
Postings loaded: 123,849 rows


In [12]:
#Clean timestamps
postings['listed_date'] = pd.to_datetime(postings['listed_time'], unit='ms')
postings['expiry_date'] = pd.to_datetime(postings['expiry'], unit='ms')

print(postings[['listed_date', 'expiry_date']].head())

          listed_date         expiry_date
0 2024-04-17 23:45:08 2024-05-17 23:45:08
1 2024-04-11 17:51:27 2024-05-11 17:51:27
2 2024-04-16 14:26:54 2024-05-16 14:26:54
3 2024-04-12 04:23:32 2024-05-12 04:23:32
4 2024-04-18 14:52:23 2024-05-18 14:52:23


In [13]:
#Extract city and state
postings['city'] = postings['location'].str.split(',').str[0].str.strip()
postings['state'] = postings['location'].str.split(',').str[1].str.strip()

print(postings[['location', 'city', 'state']].head(10))

            location           city state
0      Princeton, NJ      Princeton    NJ
1   Fort Collins, CO   Fort Collins    CO
2     Cincinnati, OH     Cincinnati    OH
3  New Hyde Park, NY  New Hyde Park    NY
4     Burlington, IA     Burlington    IA
5        Raleigh, NC        Raleigh    NC
6      United States  United States   NaN
7  San Francisco, CA  San Francisco    CA
8          Omaha, NE          Omaha    NE
9       Palm Bay, FL       Palm Bay    FL


In [15]:
#Clean salary (remove outliers)
# Use normalized_salary (already annualized)
# Remove zeros and unrealistic outliers (cap at $500k)
postings['clean_salary'] = postings['normalized_salary']
postings.loc[postings['clean_salary'] <= 0, 'clean_salary'] = np.nan
postings.loc[postings['clean_salary'] > 500000, 'clean_salary'] = np.nan

print(f"Rows with valid salary: {postings['clean_salary'].notna().sum():,}")
print(postings['clean_salary'].describe())

Rows with valid salary: 35,963
count     35963.000000
mean      94623.290934
std       57106.672169
min           1.000000
25%       51916.800000
50%       81120.000000
75%      124800.000000
max      500000.000000
Name: clean_salary, dtype: float64


In [16]:
#Clean remote column
# Fill missing remote_allowed: if location is just "United States" → likely remote
postings['is_remote'] = postings['remote_allowed'].fillna(0)
postings.loc[postings['location'].str.strip() == 'United States', 'is_remote'] = 1
postings['is_remote'] = postings['is_remote'].astype(int)

print(postings['is_remote'].value_counts())

is_remote
0    108489
1     15360
Name: count, dtype: int64


In [17]:
#Decode skills and industries
# Merge skill abbreviations → full names
job_skills_named = job_skills.merge(skills, on='skill_abr', how='left')

# Merge industry IDs → full names
job_industries_named = job_industries.merge(industries, on='industry_id', how='left')

print(job_skills_named.head())
print(job_industries_named.head())

       job_id skill_abr        skill_name
0  3884428798      MRKT         Marketing
1  3884428798        PR  Public Relations
2  3884428798       WRT   Writing/Editing
3  3887473071      SALE             Sales
4  3887465684       FIN           Finance
       job_id  industry_id                   industry_name
0  3884428798           82  Book and Periodical Publishing
1  3887473071           48                    Construction
2  3887465684           41                         Banking
3  3887467939           82  Book and Periodical Publishing
4  3887467939           80            Advertising Services


In [18]:
#Save cleaned files
postings.to_csv(OUTPUT_DIR + "postings_clean.csv", index=False)
job_skills_named.to_csv(OUTPUT_DIR + "job_skills_named.csv", index=False)
job_industries_named.to_csv(OUTPUT_DIR + "job_industries_named.csv", index=False)
benefits.to_csv(OUTPUT_DIR + "benefits_clean.csv", index=False)
companies.to_csv(OUTPUT_DIR + "companies_clean.csv", index=False)

print("✅ All cleaned files saved to /output/")

✅ All cleaned files saved to /output/
